# Question 5: SNAP enrollment and food shelf use

The ask SNAP gives a household money to buy food. A food shelf gives a household food directly. Do the two move together, or does one replace the other? The SNAP benefits may change from time-to-time, what happens to food shelf activity when the SNAP
benefits change? Does the relationship between SNAP benefits and food shelf activity differ across locations? Understanding the nature of this relationship helps The Food Group highlight the importance of SNAP to lawmakers and in turn helps them make better informed decisions.

Datasets:
| File | Grain | Role |
|---|---|---|
| `org_FSStats.xlsx` | Site × month | Monthly food-shelf visits and pounds distributed |
| `Organization.xlsx` | Organization | Partner master file (food banks, shelves, markets, agencies) |
| `Q5-snap-dashboard-data.xlsx` | County/Tribe × month (88 units, Jan 2016 – Aug 2026) | Monthly SNAP cases, persons, and net benefits issued (federal SNAP + state MFIP/MFAP food portions); joined to food-shelf data on county × month |
| `Q5-snap-dashboard-statewide-totals.xlsx` | Statewide × month (Jan 2016 – Aug 2026) | Statewide SNAP totals for trend charts and to check the county sums |


Sources:
- Minnesota Department of Children, Youth, and Families (DCYF). *SNAP Food Assistance Data Dashboard — county/Tribe monthly data* (`snap-dashboard-data-through-august-2026.xlsx`), monthly by county or Tribal Nation (accounting month), Jan 2016 – Aug 2026; household cases, persons, and food benefits issued, cancelled, and net by funding source (federal SNAP, state MFAP, MFIP food portion). https://dcyf.mn.gov/snap-food-assistance-minnesota (data through August 2026; accessed 2026-09-28)
- Minnesota Department of Children, Youth, and Families (DCYF). *SNAP Food Assistance Data Dashboard — statewide totals* (`snap-dashboard-statewide-totals-through-august-2026.xlsx`), monthly, statewide, Jan 2016 – Aug 2026; same measures as above. https://dcyf.mn.gov/snap-food-assistance-minnesota (data through August 2026; accessed 2026-09-28)

## Setup

In [6]:
# !pip install -q statsmodels openpyxl   # uncomment if needed in Colab
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Chart palette (fixed order; validated categorical slots)
C1, C2, C3, C4 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
GRID = "#e6e5e1"
plt.rcParams.update({
    "figure.figsize": (11, 4.5), "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "lines.linewidth": 2,
    "axes.titleweight": "bold", "axes.titlesize": 12,
})

### Import Data

In [7]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/MinneMUDAC 26 for BU/Suji/Data/Q5"

FS_PATH         = f"{DATA_DIR}/org_FSStats.xlsx"
ORG_PATH        = f"{DATA_DIR}/Organization.xlsx"
SNAP_PATH       = f"{DATA_DIR}/Q5-snap-dashboard-data.xlsx"
SNAP_STATE_PATH = f"{DATA_DIR}/Q5-snap-dashboard-statewide-totals.xlsx"

fs_raw   = pd.read_excel(FS_PATH)
org_raw  = pd.read_excel(ORG_PATH)
snap_raw = pd.read_excel(SNAP_PATH, parse_dates=["Report_Month"])
snap_st  = pd.read_excel(SNAP_STATE_PATH, parse_dates=["Report_Month"])

for name, df in [("org_FSStats", fs_raw), ("Organization", org_raw),
                 ("SNAP county", snap_raw), ("SNAP statewide", snap_st)]:
    print(f"{name:15s} {df.shape}")
    print("   ", list(df.columns)[:30], "\n")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
org_FSStats     (28287, 58)
    ['__ID', '_id_site', '_Year', '_Month', 'DateForSearch', 'zzCreationTS', 'zzModificationTS', 'AdultsReg', 'ChildrenReg', 'HouseholdsReg', 'IndividualsReg', 'Meals', 'Misc', 'People', 'PoundsReg', 'SeniorsReg', 'Snacks', 'UniqueVisitsReg', 'County', 'FederalDistrict', 'StateDistrict', 'FoodBank', 'ChildrenExtra', 'ChildrenProduce', 'ChildrenHoliday', 'AdultsExtra', 'AdultsProduce', 'AdultsHoliday', 'IndividualsExtra', 'SeniorsExtra'] 

Organization    (1202, 68)
    ['__ID', 'CreateTime', 'zzCreationTS', 'zzModificationTS', '_id_type', 'City', 'State', 'Zip', 'MailCity', 'MailState', 'MailZip', 'Eligibility', 'Conditions', 'MarketSeasons', 'zcountDaysOperations', 'Region', 'SuperRegion', 'County', 'StateDistrict', 'FederalDistrict', 'DistrictSection', 'AgencyType', '_id_subtype', 'AgencySubType', 'isFoodBank', 'isAgency', 'isDis

### Analysis Settings

In [8]:
# --- Food-shelf columns (confirmed in the Data Dictionary) ---
METRICS = {"HouseholdsReg": "visits", "PoundsReg": "pounds", "IndividualsReg": "individuals"}

# --- Not yet confirmed -> None = auto-detect ---
FS_DATE_COL = None      # month column in org_FSStats
ORG_KEY     = None      # join key between org_FSStats and Organization

# Food-shelf filter: regex applied to Organization.AgencyType (check value_counts in §3.1)
FOOD_SHELF_PATTERN = r"food\s*shelf|pantry"

# HouseholdsReg > IndividualsReg is impossible by definition. Flag rows with ratio >= this.
HH_INDIV_ERROR_RATIO = 2.0
# What to do with flagged rows: "null_individuals" | "null_both" | "keep"
ERROR_POLICY = "null_individuals"

# Balanced panel: keep sites that report in at least this share of months
MIN_REPORT_SHARE = 0.9

# Tribal SNAP units: "fold" into the main county, or "drop"
TRIBE_POLICY = "fold"

# Key policy date. The last EA issuance was for Feb 2023, but it was PAID in March,
# so in the dashboard's accounting months the first month without EA is April 2023 (verified in §7.1).
EA_FIRST_POST = pd.Timestamp("2023-04-01")

### Auto-detect the month col and the join key

In [9]:
def detect_date_col(df):
    # Prefer datetime columns, then names that look like a month/date
    dt_cols = [c for c in df.columns if pd.api.types.is_datetime64_any_dtype(df[c])]
    name_cols = [c for c in df.columns if re.search(r"date|month|period", str(c), re.I)
                 and not re.search(r"inactive|created|modified|update", str(c), re.I)]
    for c in dt_cols + name_cols:
        parsed = pd.to_datetime(df[c], errors="coerce")
        if parsed.notna().mean() > 0.9:
            return c
    return None

def detect_key(a, b):
    common = [c for c in a.columns if c in b.columns]
    ranked = sorted(common, key=lambda c: (not re.search(r"id$|_id|key", str(c), re.I),
                                           not re.search(r"org|agency|site", str(c), re.I)))
    for c in ranked:
        if b[c].is_unique or b[c].nunique() / len(b) > 0.95:
            return c
    return ranked[0] if ranked else None

FS_DATE_COL = FS_DATE_COL or detect_date_col(fs_raw)
ORG_KEY     = ORG_KEY or detect_key(fs_raw, org_raw)
print("FS_DATE_COL ->", FS_DATE_COL)
print("ORG_KEY     ->", ORG_KEY)
assert FS_DATE_COL and ORG_KEY, "Set FS_DATE_COL / ORG_KEY manually in §1.2"

FS_DATE_COL -> DateForSearch
ORG_KEY     -> __ID


In [11]:
# Diagnose the join key: which FS column matches which Organization column?
print("Auto-detected ORG_KEY:", ORG_KEY)
print("\norg_FSStats columns :", list(fs_raw.columns))
print("Organization columns:", list(org_raw.columns))

def norm_key(s):
    # Compare IDs as clean strings so 123 and 123.0 match
    return s.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)

id_like = lambda df: [c for c in df.columns if re.search(r"id|key|code|num|org", str(c), re.I)]
rows = []
for a in id_like(fs_raw):
    fa = norm_key(fs_raw[a]).dropna()
    for b in id_like(org_raw):
        ob = norm_key(org_raw[b]).dropna()
        rows.append({"fs_col": a, "org_col": b,
                     "fs_match_rate": fa.isin(set(ob)).mean(),
                     "org_unique": ob.is_unique})
pd.DataFrame(rows).sort_values("fs_match_rate", ascending=False).head(10)

Auto-detected ORG_KEY: __ID

org_FSStats columns : ['__ID', '_id_site', '_Year', '_Month', 'DateForSearch', 'zzCreationTS', 'zzModificationTS', 'AdultsReg', 'ChildrenReg', 'HouseholdsReg', 'IndividualsReg', 'Meals', 'Misc', 'People', 'PoundsReg', 'SeniorsReg', 'Snacks', 'UniqueVisitsReg', 'County', 'FederalDistrict', 'StateDistrict', 'FoodBank', 'ChildrenExtra', 'ChildrenProduce', 'ChildrenHoliday', 'AdultsExtra', 'AdultsProduce', 'AdultsHoliday', 'IndividualsExtra', 'SeniorsExtra', 'SeniorsProduce', 'SeniorsHoliday', 'IndividualsProduce', 'IndividualsHoliday', 'UniqueVisitsExtra', 'UniqueVisitsProduce', 'UniqueVisitsHoliday', 'UniqueVisitsTotal', 'SeniorsTotal', 'IndividualsTotal', 'ChildrenTotal', 'AdultsTotal', 'HouseholdsExtra', 'HouseholdsProduce', 'HouseholdsHoliday', 'HouseholdsTotal', 'PoundsExtra', 'PoundsProduce', 'PoundsHoliday', 'PoundsTotal', 'AgencyType', '_id_bank', 'SuperShelf', '_YearMonth', 'PoundsPerIndiv', 'IndividualsPerHH', 'id_agency', 'PoundsPerHH']
Organization

,fs_col,org_col,fs_match_rate,org_unique
117,_id_bank,__ID,1.000000,True
120,_id_bank,_id_Bank,1.000000,False
9,_id_site,__ID,0.998692,True
122,_id_bank,_id_IfFoodBank,0.995646,True
135,id_agency,__ID,0.956889,True
126,IndividualsPerHH,__ID,0.541851,True
129,IndividualsPerHH,_id_Bank,0.541851,False
127,IndividualsPerHH,_id_type,0.541769,False
131,IndividualsPerHH,_id_IfFoodBank,0.539317,True
139,id_agency,_id_Agency,0.520390,False


## Clean the food shelf data

Rules from `Data Dictionary.xlsx`
1. Use only the live **Reg** fields; Extra/Produce/Holiday/Total fields are deprecated.
2. Keep **food shelves** only (the Organization table also holds food banks and agencies).
3. Do **not** drop inactive organizations from history. Only drop months after their `InactiveDate`.
4. Fix the `HouseholdsReg > IndividualsReg` data errors.
5. Recompute the ratio columns instead of trusting the stored ones.

### Attach org attributes & Filter to food shelves

In [10]:
org_cols = [c for c in [ORG_KEY, "AgencyType", "AgencySubType", "isActive", "InactiveDate", "block_geoid"]
            if c in org_raw.columns]
county_name_cols = [c for c in org_raw.columns if re.fullmatch(r"(county|countyname|county_name)", str(c), re.I)]
org = org_raw[org_cols + county_name_cols].drop_duplicates(ORG_KEY)

fs = fs_raw.copy()
fs["month"] = pd.to_datetime(fs[FS_DATE_COL], errors="coerce").dt.to_period("M").dt.to_timestamp()
# AgencyType exists in both tables: we use the Organization master record
fs = fs.drop(columns=[c for c in org.columns if c in fs.columns and c != ORG_KEY])
fs = fs.merge(org, on=ORG_KEY, how="left", validate="many_to_one")

print("AgencyType (Organization master) — check the food-shelf filter:")
print(fs["AgencyType"].value_counts(dropna=False).to_string(), "\n")

is_shelf = fs["AgencyType"].astype(str).str.contains(FOOD_SHELF_PATTERN, case=False, regex=True)
fs = fs[is_shelf].copy()
print(f"Food-shelf rows kept: {len(fs):,} | sites: {fs[ORG_KEY].nunique():,} | "
      f"months: {fs['month'].min():%Y-%m} → {fs['month'].max():%Y-%m}")

AgencyType (Organization master) — check the food-shelf filter:
AgencyType
NaN                        27484
Agency or Tribal Nation      296
Brick and Mortar Site        194
Farmers Market               164
Outreach Program              67
Other                         38
Meal Program                  35
College Campus Program         5
Tribal Program                 3
Food Bank                      1 



/tmp/ipykernel_1364/1753532385.py:10: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  fs = fs.merge(org, on=ORG_KEY, how="left", validate="many_to_one")


ValueError: NaTType does not support strftime

### Drop only the months after a site went inactive

In [ ]:
if "InactiveDate" in fs.columns:
    inact = pd.to_datetime(fs["InactiveDate"], errors="coerce")
    after_close = inact.notna() & (fs["month"] > inact)
    print(f"Rows after InactiveDate dropped: {after_close.sum():,}")
    fs = fs[~after_close].copy()

### Metric cleaning

* Coerce to numeric; a month where **both** visits and pounds are 0 is treated as *not reported*.
* `HouseholdsReg / IndividualsReg >= HH_INDIV_ERROR_RATIO` is flagged as a data error. Individuals are self-reported (Data Catalog), so the default nulls `IndividualsReg` only. Try `ERROR_POLICY="null_both"` as a sensitivity check.

In [ ]:
for c in METRICS:
    fs[c] = pd.to_numeric(fs[c], errors="coerce")

not_reported = (fs["HouseholdsReg"].fillna(0) == 0) & (fs["PoundsReg"].fillna(0) == 0)
fs.loc[not_reported, list(METRICS)] = np.nan

ratio = fs["HouseholdsReg"] / fs["IndividualsReg"].replace(0, np.nan)
fs["flag_hh_gt_indiv"] = ratio >= HH_INDIV_ERROR_RATIO
print(f"Not-reported site-months: {not_reported.sum():,}")
print(f"Flagged HH>Indiv errors (ratio >= {HH_INDIV_ERROR_RATIO}): {fs['flag_hh_gt_indiv'].sum():,}")

if ERROR_POLICY == "null_individuals":
    fs.loc[fs["flag_hh_gt_indiv"], "IndividualsReg"] = np.nan
elif ERROR_POLICY == "null_both":
    fs.loc[fs["flag_hh_gt_indiv"], ["IndividualsReg", "HouseholdsReg"]] = np.nan

# Recompute ratios from cleaned values (do not use stored PoundsPerHH etc.)
fs["pounds_per_hh"] = fs["PoundsReg"] / fs["HouseholdsReg"]
fs["indiv_per_hh"]  = fs["IndividualsReg"] / fs["HouseholdsReg"]
fs[list(METRICS) + ["pounds_per_hh", "indiv_per_hh"]].describe().T

## Geography: map each site to a SNAP unit

`block_geoid` → first 5 digits = county FIPS → county name.

The SNAP dashboard is not a clean 87-county list, so we build a **`snap_unit`** that both sides share:

| SNAP dashboard entity | What it covers | Food-shelf counties mapped to it |
|---|---|---|
| `MNPrairie` | Dodge + Steele + Waseca consortium | Dodge, Steele, Waseca |
| `Pope` (from Jan 2022) | Western Prairie Human Services = Pope + Grant (Grant series ends Dec 2021) | Pope, Grant |
| `Mille Lacs Band`, `Red Lake Nation`, `White Earth Band` | Tribal Nations administer their own SNAP | `TRIBE_POLICY="fold"` adds them to Mille Lacs, Beltrami, Mahnomen |

All other units are single counties.

In [ ]:
# Minnesota counties in DHS county-code order (code n -> FIPS 27 + (2n-1)), spelled as in the SNAP dashboard
MN_COUNTIES = ["Aitkin","Anoka","Becker","Beltrami","Benton","Big Stone","Blue Earth","Brown","Carlton","Carver",
    "Cass","Chippewa","Chisago","Clay","Clearwater","Cook","Cottonwood","Crow Wing","Dakota","Dodge","Douglas",
    "Faribault","Fillmore","Freeborn","Goodhue","Grant","Hennepin","Houston","Hubbard","Isanti","Itasca","Jackson",
    "Kanabec","Kandiyohi","Kittson","Koochiching","Lac Qui Parle","Lake","Lake of Woods","Le Sueur","Lincoln","Lyon",
    "McLeod","Mahnomen","Marshall","Martin","Meeker","Mille Lacs","Morrison","Mower","Murray","Nicollet","Nobles",
    "Norman","Olmsted","Otter Tail","Pennington","Pine","Pipestone","Polk","Pope","Ramsey","Red Lake","Redwood",
    "Renville","Rice","Rock","Roseau","St. Louis","Scott","Sherburne","Sibley","Stearns","Steele","Stevens","Swift",
    "Todd","Traverse","Wabasha","Wadena","Waseca","Washington","Watonwan","Wilkin","Winona","Wright","Yellow Medicine"]
assert len(MN_COUNTIES) == 87
FIPS_TO_COUNTY = {f"27{2*i+1:03d}": name for i, name in enumerate(MN_COUNTIES)}

COUNTY_TO_UNIT = {c: c for c in MN_COUNTIES}
COUNTY_TO_UNIT.update({"Dodge": "MNPrairie", "Steele": "MNPrairie", "Waseca": "MNPrairie", "Grant": "Pope"})
TRIBE_TO_COUNTY = {"Mille Lacs Band": "Mille Lacs", "Red Lake Nation": "Beltrami", "White Earth Band": "Mahnomen"}

METRO7 = {"Anoka", "Carver", "Dakota", "Hennepin", "Ramsey", "Scott", "Washington"}

def norm_county(s):
    # Normalize free-text county names to the dashboard spelling
    s = str(s).strip().replace(" County", "").title()
    fixes = {"Lac Qui Parle": "Lac Qui Parle", "Lake Of The Woods": "Lake of Woods", "Lake Of Woods": "Lake of Woods",
             "Saint Louis": "St. Louis", "St Louis": "St. Louis", "St. Louis": "St. Louis", "Mcleod": "McLeod"}
    return fixes.get(s, s)

geoid = fs["block_geoid"].astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(15) \
        if "block_geoid" in fs.columns else pd.Series(pd.NA, index=fs.index, dtype="string")
fs["county"] = geoid.str[:5].map(FIPS_TO_COUNTY)
if county_name_cols:   # fallback when block_geoid is blank
    fs["county"] = fs["county"].fillna(fs[county_name_cols[0]].map(norm_county))
fs["county"] = fs["county"].where(fs["county"].isin(MN_COUNTIES))
fs["snap_unit"] = fs["county"].map(COUNTY_TO_UNIT)

unmatched = fs.loc[fs["snap_unit"].isna(), ORG_KEY].nunique()
print(f"Sites without a Minnesota county: {unmatched} (excluded from the join)")
fs = fs[fs["snap_unit"].notna()].copy()
print(f"Sites kept: {fs[ORG_KEY].nunique():,} across {fs['snap_unit'].nunique()} SNAP units")

## SNAP & Build the county x month panel

### SNAP variables

* `snap_persons` = `PersonCount` → **enrollment**
* `snap_benefits` = `Total_Net` (federal SNAP + state MFAP + MFIP food portion, net of cancellations). We use the total because `PersonCount` covers all three.
* `benefit_pp` = `snap_benefits / snap_persons` → **benefit generosity**. This is what dropped when Emergency Allotments ended.

In [ ]:
snap = snap_raw.rename(columns={"Report_Month": "month", "CountyTribeName": "entity"}).copy()
snap["month"] = snap["month"].dt.to_period("M").dt.to_timestamp()

if TRIBE_POLICY == "fold":
    snap["snap_unit"] = snap["entity"].replace(TRIBE_TO_COUNTY)
else:
    snap = snap[~snap["entity"].isin(TRIBE_TO_COUNTY)]
    snap["snap_unit"] = snap["entity"]

snap = (snap.groupby(["snap_unit", "month"], as_index=False)
            .agg(snap_cases=("CaseCount", "sum"), snap_persons=("PersonCount", "sum"),
                 snap_benefits=("Total_Net", "sum"), snap_fed_net=("SNAP_Federal_Food_Net", "sum")))
snap["benefit_pp"] = snap["snap_benefits"] / snap["snap_persons"]

# Check: county sums vs statewide file
chk = snap.groupby("month")[["snap_persons", "snap_benefits"]].sum().join(
      snap_st.set_index(snap_st["Report_Month"].dt.to_period("M").dt.to_timestamp())[["PersonCount", "Total_Net"]])
print("Max abs % gap county-sum vs statewide (persons, benefits):",
      ((chk["snap_persons"] / chk["PersonCount"] - 1).abs().max() * 100).round(2),
      ((chk["snap_benefits"] / chk["Total_Net"] - 1).abs().max() * 100).round(2))

### Aggregate food shelves to SNAP unit x month

Food-shelf totals can rise just because **new sites start reporting**. We therefore build two versions:
* `all`: every reporting site
* `balanced`: only sites reporting in ≥ `MIN_REPORT_SHARE` of months. This is the **default for inference**.

In [ ]:
months_all = fs["month"].nunique()
share = fs.dropna(subset=["HouseholdsReg"]).groupby(ORG_KEY)["month"].nunique() / months_all
balanced_sites = share[share >= MIN_REPORT_SHARE].index
print(f"Balanced-panel sites: {len(balanced_sites):,} of {fs[ORG_KEY].nunique():,}")

def agg_fs(df):
    return (df.groupby(["snap_unit", "month"], as_index=False)
              .agg(visits=("HouseholdsReg", "sum"), pounds=("PoundsReg", "sum"),
                   individuals=("IndividualsReg", "sum"), n_sites=(ORG_KEY, "nunique")))

fs_all = agg_fs(fs.dropna(subset=["HouseholdsReg"]))
fs_bal = agg_fs(fs[fs[ORG_KEY].isin(balanced_sites)].dropna(subset=["HouseholdsReg"]))

### Join with SNAP

In [ ]:
def build_panel(fs_agg):
    p = fs_agg.merge(snap, on=["snap_unit", "month"], how="inner", validate="one_to_one")
    p["metro"] = p["snap_unit"].isin(METRO7).astype(int)
    p["moy"] = p["month"].dt.month                      # month-of-year (seasonality)
    for c in ["visits", "pounds", "individuals", "snap_persons", "benefit_pp"]:
        p[f"ln_{c}"] = np.log(p[c].where(p[c] > 0))
    # Year-over-year log change (removes seasonality without needing pre-2022 food-shelf data)
    p = p.sort_values(["snap_unit", "month"])
    for c in ["visits", "pounds", "individuals", "snap_persons", "benefit_pp"]:
        lag12 = p.groupby("snap_unit")[f"ln_{c}"].shift(12)
        same_gap = p.groupby("snap_unit")["month"].shift(12) == p["month"] - pd.DateOffset(months=12)
        p[f"yoy_{c}"] = (p[f"ln_{c}"] - lag12).where(same_gap)
    return p

panel_all = build_panel(fs_all)
panel     = build_panel(fs_bal)      # default
print(f"Panel: {panel['snap_unit'].nunique()} units × {panel['month'].nunique()} months = {len(panel):,} rows")
print(f"Coverage: {panel['month'].min():%Y-%m} → {panel['month'].max():%Y-%m}")
panel.head()

Q5 - A: Do SNAP and food shelves move together, or does one replace the other?